# The Waystation: reviewed AI-to-Scripture pipeline

This public notebook documents the contract used by the playable submission. The game sends an authored vignette ID—not arbitrary player text. Gloo AI Studio must select a need and passage through a constrained tool call; the server validates the pair; YouVersion supplies the displayed text. No credentials are stored here.

In [ ]:
VIGNETTES = {
    'mara_grief': {
        'needs': {'comfort', 'presence'},
        'passages': {'comfort': ['PSA.34.18'], 'presence': ['PSA.23.4']},
    },
    'oren_weariness': {
        'needs': {'rest', 'courage'},
        'passages': {'rest': ['MAT.11.28-30'], 'courage': ['ISA.40.31']},
    },
    'fen_belonging': {
        'needs': {'belonging', 'mercy'},
        'passages': {'belonging': ['GAL.3.28'], 'mercy': ['LUK.6.36']},
    },
}

def validate_selection(vignette_id, selection):
    vignette = VIGNETTES[vignette_id]
    need = selection['need_id']
    passage = selection['passage_id']
    reflection = selection['reflection'].strip()
    return (
        need in vignette['needs']
        and passage in vignette['passages'][need]
        and 0 < len(reflection) <= 140
        and not any(mark in reflection for mark in ['\"', '“', '”'])
    )

reviewed_fixture = {
    'need_id': 'comfort',
    'passage_id': 'PSA.34.18',
    'reflection': 'Some grief cannot be carried away, but it need not be carried alone.',
}
assert validate_selection('mara_grief', reviewed_fixture)
reviewed_fixture

## Live request shape

The production Rust service exchanges the Gloo client credentials for a one-hour bearer token and posts to `https://platform.ai.gloo.com/ai/v2/chat/completions`. The `select_remembrance` function schema enumerates only the current vignette's allowed need and passage IDs, with `tool_choice: required`. After independent validation, it requests `https://api.youversion.com/v1/bibles/3034/passages/{passage_id}` using the server-only `X-YVP-App-Key`.

The final response includes the Gloo model and routing mechanism plus `scripture_source = you_version_live | cache | fixture`. The game shows that provenance on every completed card.

In [ ]:
# A deliberately invalid cross-theme selection is rejected before any Scripture lookup.
invalid = {
    'need_id': 'rest',
    'passage_id': 'MAT.11.28-30',
    'reflection': 'Rest now.',
}
assert not validate_selection('mara_grief', invalid)
print('Allowlist validation passed.')